# **PIANO AMT (Onsets & Frames) — Demo**



# Setup

In [ ]:
# #@title 0. Configuration
# REPO_URL = "https://github.com/Mobinmo83/AMT.git"
# REPO_BRANCH = "main"
# REPO_CLONE_DIR = "/content/repo"

# HF_REPO_ID_OVERRIDE = "Mobinmo83/piano-amt-demo"
# HF_CHECKPOINT_FILENAME_OVERRIDE = "checkpoints/best.pt"
# MODEL_COMPLEXITY_OVERRIDE = 48

In [4]:
#@title 0. Configuration
from getpass import getpass

GITHUB_TOKEN = getpass("GitHub token: ")

REPO_URL = f"https://{GITHUB_TOKEN}@github.com/Mobinmo83/AMT_FYP.git"
REPO_CLONE_DIR = "/content/repo"
PROJECT_DIR = f"{REPO_CLONE_DIR}/piano_amt"

HF_REPO_ID_OVERRIDE = "Mobinmo83/piano-amt-demo"
HF_CHECKPOINT_FILENAME_OVERRIDE = "checkpoints/best.pt"
MODEL_COMPLEXITY_OVERRIDE = 48

GitHub token: ··········


In [3]:
# #@title 1. Cloning repo and installing public demo requirements
# import os
# import shutil
# from pathlib import Path

# repo_dir = Path(REPO_CLONE_DIR)
# if repo_dir.exists():
#     shutil.rmtree(repo_dir)

# !git clone -q --branch "$REPO_BRANCH" "$REPO_URL" "$REPO_CLONE_DIR"
# %cd $REPO_CLONE_DIR

# # FluidSynth + GM soundfont improve predicted MIDI playback in Colab.
# !apt-get -qq update
# !apt-get -qq install -y fluidsynth fluid-soundfont-gm > /dev/null
# !pip -q install -r requirements_demo.txt huggingface_hub


fatal: Remote branch main not found in upstream origin
[Errno 2] No such file or directory: '/content/repo/piano_amt'
/content
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements_demo.txt'


In [5]:
#@title 1. Cloning repo and installing public demo requirements
import os
import shutil
from pathlib import Path

repo_dir = Path(REPO_CLONE_DIR)
if repo_dir.exists():
    shutil.rmtree(repo_dir)

# Clone default branch automatically. Do not force --branch main.
!git clone -q "$REPO_URL" "$REPO_CLONE_DIR"

# Clear token variable after clone
GITHUB_TOKEN = None
REPO_URL = None

%cd $PROJECT_DIR

# FluidSynth + GM soundfont improve predicted MIDI playback in Colab.
!apt-get -qq update
!apt-get -qq install -y fluidsynth fluid-soundfont-gm > /dev/null

!pip -q install -r requirements_demo.txt huggingface_hub

print("Setup complete ✓")

/content/repo/piano_amt
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 105.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.7/253.7 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 98.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 83.0 MB/s eta 0:00:00
Setup complete ✓


In [6]:
#@title 2. Imports and runtime setup
import json
import os
import sys
import uuid
import shutil
from pathlib import Path

import pandas as pd
import pretty_midi
import torch
import ipywidgets as widgets
from google.colab import files
from huggingface_hub import hf_hub_download
from IPython.display import Audio, FileLink, HTML, Markdown, display

PROJECT_ROOT = REPO_CLONE_DIR

os.environ["AMT_DEMO_HF_REPO_ID"] = HF_REPO_ID_OVERRIDE
os.environ["AMT_DEMO_HF_CHECKPOINT_FILENAME"] = HF_CHECKPOINT_FILENAME_OVERRIDE
os.environ["AMT_DEMO_MODEL_COMPLEXITY"] = str(MODEL_COMPLEXITY_OVERRIDE)
os.environ["AMT_DEMO_REPO_ROOT"] = PROJECT_ROOT

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from demo.checkpoint import load_demo_model, format_checkpoint_summary, maybe_torchinfo_summary
from demo.decoder_presets import (
    DEFAULT_MODE,
    config_table_dict,
    get_decoder_preset,
    list_decoder_modes,
    make_decoder_config,
)
from demo.demo_config import AUDIO_DIR, HTML_DIR, MIDI_DIR, PLOT_DIR, ensure_demo_dirs
from demo.evaluation import (
    compare_decoder_modes,
    decoder_config_table,
    evaluate_prediction_vs_gt,
    main_scores_table,
    supplementary_table,
)
from demo.inference import (
    gt_rolls_to_note_events,
    prediction_to_note_events,
    run_model_on_mel,
    save_gt_eval_midi,
    save_prediction_midi,
)
from demo.rendering import (
    PIANO_PROGRAMS,
    plot_midi_with_sustain_and_velocity,
    plot_note_events_colored,
    plot_piano_roll_side_by_side,
    plot_pred_vs_gt_events,
    plot_roll_diff,
    render_visual_midi,
    synthesize_and_save,
    synthesize_pretty_midi,
)
from demo.sources import (
    audio_path_to_mel,
    list_demo_sample_names,
    load_ground_truth_labels,
    resolve_demo_sample_paths,
    save_uploaded_audio,
)

ensure_demo_dirs()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


Device: cuda
GPU: Tesla T4


In [7]:
#@title 3. Download demo assets from Hugging Face
PROJECT_ROOT_PATH = Path(PROJECT_ROOT)
LOCAL_MANIFEST = PROJECT_ROOT_PATH / "demo" / "sample_manifest.json"

manifest_cache_path = hf_hub_download(
    repo_id=HF_REPO_ID_OVERRIDE,
    filename="demo/sample_manifest.json",
)
LOCAL_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
shutil.copy2(manifest_cache_path, LOCAL_MANIFEST)

with open(LOCAL_MANIFEST, "r", encoding="utf-8") as f:
    manifest = json.load(f)

# Download the public demo assets used by the notebook.  The original MIDI is
# included for listening/display/download; metric ground truth still comes from
# the cached label rolls to match the final evaluation notebook.
for sample in manifest["samples"]:
    for key in ["audio", "labels", "midi", "original_midi"]:
        remote_path = sample.get(key)
        if not remote_path:
            continue
        cached_file = hf_hub_download(
            repo_id=HF_REPO_ID_OVERRIDE,
            filename=remote_path,
        )
        local_target = PROJECT_ROOT_PATH / remote_path
        local_target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(cached_file, local_target)

print(f"Downloaded manifest and {len(manifest['samples'])} public demo sample(s).")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


sample_manifest.json: 0.00B [00:00, ?B/s]

demo_assets/sample_01.wav:   0%|          | 0.00/100M [00:00<?, ?B/s]

demo_assets/sample_01_labels.npz:   0%|          | 0.00/93.4k [00:00<?, ?B/s]

sample_01_original.mid:   0%|          | 0.00/65.8k [00:00<?, ?B/s]

...

sample_05_original.mid:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

demo_assets/sample_06.wav:   0%|          | 0.00/24.7M [00:00<?, ?B/s]

demo_assets/sample_06_labels.npz:   0%|          | 0.00/31.1k [00:00<?, ?B/s]

sample_06_original.mid:   0%|          | 0.00/13.8k [00:00<?, ?B/s]

Downloaded manifest and 6 public demo sample(s).


# Initialize Model

In [8]:

#@title 4. Loading checkpoint
MODEL, CKPT, CKPT_PATH = load_demo_model(DEVICE)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


checkpoints/best.pt:   0%|          | 0.00/318M [00:00<?, ?B/s]

In [9]:
#@title 5. Model info
Model_info = True  #@param {type:"boolean"}

if Model_info:
    print(format_checkpoint_summary(MODEL, CKPT, CKPT_PATH))
    print("Compact torchinfo summary:")
    print(maybe_torchinfo_summary(MODEL))
    print("Full architecture:")
    print(MODEL)

Checkpoint: /content/amt_demo/checkpoints/checkpoints/best.pt
Epoch: 990
Val loss: 0.045293346999117925
Best val loss: 0.045293346999117925
Global step: 118800
Trainable params: 26,491,256
Compact torchinfo summary:
Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
OnsetsAndFrames                          [1, 229, 512]             [1, 512, 88]              --
├─Sequential: 1-1                        [1, 229, 512]             [1, 512, 88]              --
│    └─ConvStack: 2-1                    [1, 229, 512]             [1, 512, 768]             --
│    │    └─Sequential: 3-1              [1, 1, 512, 229]          [1, 96, 512, 57]          63,216
│    │    └─Sequential: 3-2              [1, 512, 5472]            [1, 512, 768]             4,203,264
│    └─BiLSTM: 2-2                       [1, 512, 768]             [1, 512, 768]             --
│    │    └─LSTM: 3-3                    [1, 512, 768]             [1, 512, 768]             3,5

# Upload audio

In [32]:

#@title 6. Upload Audio or choose from test set
DEMO_SAMPLE_NAMES = list_demo_sample_names()
SOURCE_MODE = "Upload custom audio" #@param ["Public demo sample", "Upload custom audio"]
DEMO_SAMPLE_INDEX = 1 #@param {type:"integer"}

print("Source mode:", SOURCE_MODE)
if SOURCE_MODE == "Public demo sample":
    if not DEMO_SAMPLE_NAMES:
        raise RuntimeError("No public demo samples were found.")
    if DEMO_SAMPLE_INDEX < 0 or DEMO_SAMPLE_INDEX >= len(DEMO_SAMPLE_NAMES):
        raise IndexError(f"DEMO_SAMPLE_INDEX must be between 0 and {len(DEMO_SAMPLE_NAMES)-1}")
    print("Selected sample:", DEMO_SAMPLE_NAMES[DEMO_SAMPLE_INDEX])
else:
    print("You will upload an audio file in the next cell.")




Source mode: Upload custom audio
You will upload an audio file in the next cell.


In [33]:
#@title 7. Preparing input audio
from pathlib import Path

original_midi_path = None
metadata = {}

if SOURCE_MODE == "Public demo sample":
    selected_name = DEMO_SAMPLE_NAMES[DEMO_SAMPLE_INDEX]
    audio_path, labels_path, original_midi_path, metadata = resolve_demo_sample_paths(selected_name, PROJECT_ROOT)

    mel = audio_path_to_mel(audio_path)
    gt = load_ground_truth_labels(labels_path)
    title = Path(audio_path).stem

    source_type = "demo_sample"
    print("Prepared public demo sample:", title)
    if original_midi_path is not None:
        print("Original MAESTRO MIDI:", original_midi_path)
else:
    uploaded = files.upload()
    if not uploaded:
        raise ValueError("Please upload an audio file.")

    first_name = next(iter(uploaded.keys()))
    audio_path = save_uploaded_audio(uploaded[first_name], first_name)
    mel = audio_path_to_mel(audio_path)
    gt = None
    title = Path(audio_path).stem

    source_type = "custom_upload"
    print("Prepared uploaded audio:", title)

print("Audio path:", audio_path)
print("Mel shape:", tuple(mel.shape))
if metadata:
    display(pd.DataFrame([metadata]).T.rename(columns={0: "Value"}))


Saving Waltz No. 19 in A Minor, Op_16k_mono.wav to Waltz No. 19 in A Minor, Op_16k_mono.wav
Prepared uploaded audio: Waltz No. 19 in A Minor, Op_16k_mono_16k_mono
Audio path: /content/amt_demo/uploaded_audio/Waltz No. 19 in A Minor, Op_16k_mono_16k_mono.wav
Mel shape: (229, 4147)


In [36]:
#@title 8. Transcription and MIDI decoding
# This cell runs the trained model, decodes the output rolls into MIDI,
# and saves the predicted MIDI/audio/plots.
#
# Three dissertation demo modes:
#   baseline  = adv_baseline, tuned baseline with no post-processing
#   m3_m4     = adv_m3_m4, minimum note duration + duplicate removal
#   m2_m3_m4  = adv_m2_m3_m4, frame smoothing + minimum duration + duplicate removal
#
# The onset/frame thresholds below are prediction thresholds and are used
# for all three modes. Ground-truth cached label rolls remain decoded at 0.5/0.5.

import pandas as pd

# ── Main demo choice ────────────────────────────────────────────────────────
DECODE_PROFILE = "baseline" #@param ["baseline", "m3_m4", "m2_m3_m4"]
PIANO_SOUND = "Acoustic Grand" #@param ["Acoustic Grand", "Bright Acoustic", "Electric Grand", "Honky-tonk", "Electric Piano"]

# ── Shared prediction thresholds for all modes ─────────────────────────────
# Report-final tuned values: 0.40 / 0.40
ONSET_THRESHOLD = 0.40 #@param {type:"number"}
FRAME_THRESHOLD = 0.40 #@param {type:"number"}

# ── M2 settings: used only by m2_m3_m4 ─────────────────────────────────────
# Report-final quality setting: closing, kernel=3
FRAME_SMOOTHING_KERNEL_M2 = 3 #@param {type:"integer"}
FRAME_SMOOTHING_METHOD_M2 = "closing" #@param ["closing", "median", "gaussian"]

# ── M3 setting: used by m3_m4 and m2_m3_m4 ─────────────────────────────────
# Report-final value: 55 ms
MIN_NOTE_DURATION_MS_M3 = 55.0 #@param {type:"number"}

# ── M4 setting: used by m3_m4 and m2_m3_m4 ─────────────────────────────────
# Report-final value: 0.06 s = 60 ms
DUPLICATE_TOLERANCE_SEC_M4 = 0.06 #@param {type:"number"}


# ── Build active decoder configuration ─────────────────────────────────────
if DECODE_PROFILE == "baseline":
    cfg = make_decoder_config(
        "baseline",
        onset_threshold=float(ONSET_THRESHOLD),
        frame_threshold=float(FRAME_THRESHOLD),
    )

    used_params = {
        "Mode": "adv_baseline",
        "Description": "Tuned baseline — no post-processing",
        "Decoder type": "baseline rolls_to_note_events",
        "Onset threshold (baseline/advanced prediction decode)": float(ONSET_THRESHOLD_BASELINE_AND_ADVANCED),
        "Frame threshold (baseline/advanced prediction decode)": float(FRAME_THRESHOLD_BASELINE_AND_ADVANCED),
        "Post-processing methods": "None",
        "Ground-truth decode": "cached label rolls at 0.5/0.5",
    }

elif DECODE_PROFILE == "m3_m4":
    cfg = make_decoder_config(
        "efficient_m3_m4",
        onset_threshold=float(ONSET_THRESHOLD),
        frame_threshold=float(FRAME_THRESHOLD),
        min_note_duration_ms=float(MIN_NOTE_DURATION_MS_M3),
        use_duplicate_removal=True,
        duplicate_tolerance_sec=float(DUPLICATE_TOLERANCE_SEC_M4),
    )

    used_params = {
        "Mode": "adv_m3_m4",
        "Description": "Efficient mode — M3 + M4",
        "Decoder type": "advanced_rolls_to_note_events",
        "Onset threshold (baseline/advanced prediction decode)": float(ONSET_THRESHOLD),
        "Frame threshold (baseline/advanced prediction decode)": float(FRAME_THRESHOLD),
        "Minimum note duration (M3, ms)": float(MIN_NOTE_DURATION_MS_M3),
        "Duplicate removal (M4)": True,
        "Duplicate tolerance (M4, s)": float(DUPLICATE_TOLERANCE_SEC_M4),
        "Ground-truth decode": "cached label rolls at 0.5/0.5",
    }

elif DECODE_PROFILE == "m2_m3_m4":
    cfg = make_decoder_config(
        "quality_m2_m3_m4",
        onset_threshold=float(ONSET_THRESHOLD),
        frame_threshold=float(FRAME_THRESHOLD),
        use_frame_smoothing=True,
        frame_smoothing_kernel=int(FRAME_SMOOTHING_KERNEL_M2),
        frame_smoothing_method=str(FRAME_SMOOTHING_METHOD_M2),
        min_note_duration_ms=float(MIN_NOTE_DURATION_MS_M3),
        use_duplicate_removal=True,
        duplicate_tolerance_sec=float(DUPLICATE_TOLERANCE_SEC_M4),
    )

    used_params = {
        "Mode": "adv_m2_m3_m4",
        "Description": "Quality mode — M2 + M3 + M4",
        "Decoder type": "advanced_rolls_to_note_events",
        "Onset threshold (baseline/advanced prediction decode)": float(ONSET_THRESHOLD),
        "Frame threshold (baseline/advanced prediction decode)": float(FRAME_THRESHOLD),
        "Frame smoothing (M2)": True,
        "Smoothing method (M2)": str(FRAME_SMOOTHING_METHOD_M2),
        "Smoothing kernel (M2)": int(FRAME_SMOOTHING_KERNEL_M2),
        "Minimum note duration (M3, ms)": float(MIN_NOTE_DURATION_MS_M3),
        "Duplicate removal (M4)": True,
        "Duplicate tolerance (M4, s)": float(DUPLICATE_TOLERANCE_SEC_M4),
    }

else:
    raise ValueError(f"Unknown DECODE_PROFILE: {DECODE_PROFILE}")


display(Markdown("### Active decoder configuration"))
display(Markdown(
    f"**Selected demo option:** `{DECODE_PROFILE}`  \n"
    f"**Internal evaluation name:** `{cfg.name}`  \n"
    f"**Piano sound:** `{PIANO_SOUND}`"
))

display(pd.DataFrame(
    [{"Parameter": k, "Value": v} for k, v in used_params.items()]
))


# ── Run full-length model inference ─────────────────────────────────────────
pred = run_model_on_mel(MODEL, mel, DEVICE)

print("Prediction tensor shapes:")
for k, v in pred.items():
    print(f"  {k}: {tuple(v.shape)}")


# ── Output paths ────────────────────────────────────────────────────────────
run_id = f"{title}_{cfg.name}_{uuid.uuid4().hex[:8]}"

pred_midi_path = MIDI_DIR / f"{run_id}_predicted.mid"
pred_wav_path = AUDIO_DIR / f"{run_id}_predicted.wav"
pred_png_path = PLOT_DIR / f"{run_id}_pred_roll.png"
pred_events_png_path = PLOT_DIR / f"{run_id}_pred_events.png"
pred_html_path = HTML_DIR / f"{run_id}_pred_visual_midi.html"

gt_eval_midi_path = None
gt_eval_wav_path = None
gt_png_path = None
diff_png_path = None
event_overlay_path = None
original_midi_wav_path = None
original_sustain_png_path = None
original_html_path = None


# ── Save predicted MIDI/audio/Visual MIDI ───────────────────────────────────
program = PIANO_PROGRAMS.get(PIANO_SOUND, 0)

pred_midi_path = save_prediction_midi(
    pred,
    pred_midi_path,
    decoder_config=cfg,
    program=program,
)

pred_pm = pretty_midi.PrettyMIDI(str(pred_midi_path))
pred_wav_path = synthesize_and_save(pred_pm, pred_wav_path, piano_sound=PIANO_SOUND)
render_visual_midi(pred_pm, html_path=pred_html_path, show_inline=False)


# ── If GT exists, save evaluation-GT MIDI reconstructed from cached rolls ───
if gt is not None:
    gt_eval_midi_path = MIDI_DIR / f"{run_id}_ground_truth_eval_rolls.mid"
    gt_eval_wav_path = AUDIO_DIR / f"{run_id}_ground_truth_eval_rolls.wav"
    gt_png_path = PLOT_DIR / f"{run_id}_ground_truth_and_pred_rolls.png"
    diff_png_path = PLOT_DIR / f"{run_id}_diff_roll.png"
    event_overlay_path = PLOT_DIR / f"{run_id}_pred_vs_gt_events.png"

    save_gt_eval_midi(gt, gt_eval_midi_path, program=program)
    gt_eval_pm = pretty_midi.PrettyMIDI(str(gt_eval_midi_path))
    synthesize_and_save(gt_eval_pm, gt_eval_wav_path, piano_sound=PIANO_SOUND)


# ── If original MAESTRO MIDI exists, save playback/rendering version ────────
# Original MIDI is for listening, download, sustain/velocity display, and
# qualitative comparison. Metrics use cached GT label rolls, not original MIDI.
if original_midi_path is not None and Path(original_midi_path).exists():
    original_pm = pretty_midi.PrettyMIDI(str(original_midi_path))
    original_midi_wav_path = AUDIO_DIR / f"{run_id}_original_maestro_midi.wav"
    original_sustain_png_path = PLOT_DIR / f"{run_id}_original_velocity_sustain.png"
    original_html_path = HTML_DIR / f"{run_id}_original_visual_midi.html"

    synthesize_and_save(original_pm, original_midi_wav_path, piano_sound=PIANO_SOUND)
    render_visual_midi(original_pm, html_path=original_html_path, show_inline=False)


print("Saved predicted MIDI:", pred_midi_path)
print("Saved predicted WAV:", pred_wav_path)

if gt_eval_midi_path is not None:
    print("Saved evaluation-GT MIDI:", gt_eval_midi_path)

if original_midi_path is not None:
    print("Original MAESTRO MIDI:", original_midi_path)

### Active decoder configuration

**Selected demo option:** `baseline`  
**Internal evaluation name:** `adv_baseline`  
**Piano sound:** `Acoustic Grand`

,Parameter,Value
0,Mode,adv_baseline
1,Description,Tuned baseline — no post-processing
2,Decoder type,baseline rolls_to_note_events
3,Onset threshold (baseline/advanced prediction ...,0.4
4,Frame threshold (baseline/advanced prediction ...,0.4
5,Post-processing methods,None
6,Ground-truth decode,cached label rolls at 0.5/0.5


Prediction tensor shapes:
  onset: (4147, 88)
  frame: (4147, 88)
  offset: (4147, 88)
  velocity: (4147, 88)


AttributeError: unexpected attribute 'render_mode' to Label, similar attributes are renderers

# Prediction and Inference


In [ ]:
#@title 9. Prediction results
import matplotlib.pyplot as plt

display(Markdown(f"## Results — {title}"))
display(Markdown(f"**Source:** `{source_type}`"))
display(Markdown(f"**Decoder mode:** `{DECODER_MODE}` → `{cfg.name}`"))
display(Markdown(f"**Piano sound:** `{PIANO_SOUND}`"))

display(Markdown("### Original audio"))
display(Audio(filename=str(audio_path)))

display(Markdown("### Predicted MIDI audio"))
display(Audio(filename=str(pred_wav_path)))

display(Markdown("### Predicted MIDI note events"))
pred_events = prediction_to_note_events(pred, decoder_config=cfg)
pred_events_fig = plot_note_events_colored(
    pred_events,
    title=f"Predicted note events — {cfg.name}",
    save_path=pred_events_png_path,
)
display(pred_events_fig)
plt.close(pred_events_fig)

display(Markdown("### Predicted piano roll"))
pred_fig = plot_piano_roll_side_by_side(
    pred_frame=pred["frame"],
    gt_frame=None,
    title=f"Prediction — {title}",
    save_path=pred_png_path,
    frame_threshold=cfg.frame_threshold,
)
display(pred_fig)
plt.close(pred_fig)

display(Markdown("### Predicted Visual MIDI"))
pred_visual = render_visual_midi(pred_pm, html_path=pred_html_path, show_inline=True)
if pred_visual is None:
    display(Markdown(f"Visual MIDI HTML saved here: `{pred_html_path}`"))


In [31]:
#@title 10. Original audio and evaluation (test-set demo samples only)
import matplotlib.pyplot as plt

if gt is None:
    display(Markdown("> Uploaded custom audio has no ground-truth label rolls, so evaluation and original MIDI comparison are skipped."))
else:
    display(Markdown("### Evaluation protocol"))
    display(Markdown(
        "Metrics use the cached ground-truth label rolls decoded with the same GT path used in the final validation/test evaluation. "
        "The original MAESTRO MIDI is displayed for listening and download, but it is not used as the metric reference."
    ))

    if original_midi_path is not None and Path(original_midi_path).exists():
        display(Markdown("### Original MAESTRO MIDI audio"))
        display(Audio(filename=str(original_midi_wav_path)))

        display(Markdown("### Original MAESTRO MIDI: velocity and sustain"))
        original_fig = plot_midi_with_sustain_and_velocity(
            original_midi_path,
            title=f"Original MAESTRO MIDI — {title}",
            save_path=original_sustain_png_path,
        )
        display(original_fig)
        plt.close(original_fig)

        display(Markdown("### Original MAESTRO Visual MIDI"))
        original_visual = render_visual_midi(original_pm, html_path=original_html_path, show_inline=True)
        if original_visual is None:
            display(Markdown(f"Original Visual MIDI HTML saved here: `{original_html_path}`"))

    display(Markdown("### Evaluation ground-truth MIDI reconstructed from cached rolls"))
    display(Audio(filename=str(gt_eval_wav_path)))

    results = evaluate_prediction_vs_gt(pred, gt, decoder_config=cfg)

    display(Markdown("### Predicted vs evaluation ground-truth piano rolls"))
    comp_fig = plot_piano_roll_side_by_side(
        pred_frame=pred["frame"],
        gt_frame=gt["frame"],
        title=title,
        save_path=gt_png_path,
        frame_threshold=cfg.frame_threshold,
    )
    display(comp_fig)
    plt.close(comp_fig)

    diff_fig = plot_roll_diff(
        pred_frame=pred["frame"],
        gt_frame=gt["frame"],
        frame_threshold=cfg.frame_threshold,
        save_path=diff_png_path,
    )
    display(diff_fig)
    plt.close(diff_fig)

    display(Markdown("### Predicted vs evaluation ground-truth note events"))
    overlay_fig = plot_pred_vs_gt_events(
        results["events"]["pred"],
        results["events"]["gt_eval"],
        title=f"Prediction vs GT eval-roll events — {cfg.name}",
        max_time=60,
        save_path=event_overlay_path,
    )
    display(overlay_fig)
    plt.close(overlay_fig)

    display(Markdown("### Quantitative evaluation"))
    display(main_scores_table(results).style.format({
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1": "{:.4f}",
    }))

    display(Markdown("### Supplementary / failure-case analysis"))
    display(supplementary_table(results).style.format({"Value": "{:.4f}"}))

    display(Markdown("### Baseline vs efficient vs quality decoder comparison"))
    comparison = compare_decoder_modes(pred, gt, modes=["baseline", "efficient_m3_m4", "quality_m2_m3_m4"])
    display(comparison.style.format({
        "Frame F1": "{:.4f}",
        "Note F1": "{:.4f}",
        "N+Off F1": "{:.4f}",
        "N+Off+Vel F1": "{:.4f}",
    }))


### Evaluation protocol

Metrics use the cached ground-truth label rolls decoded with the same GT path used in the final validation/test evaluation. The original MAESTRO MIDI is displayed for listening and download, but it is not used as the metric reference.

### Original MAESTRO MIDI audio

FileNotFoundError: [Errno 2] No such file or directory: 'None'

In [15]:
#@title 11 downloading files # add orginal audio and predicted audio downlaod as well
print("Download files")

download_items = [
    ("Predicted selected-mode MIDI", pred_midi_path),
    ("Predicted selected-mode audio WAV", pred_wav_path),
    ("Predicted piano-roll PNG", pred_png_path),
    ("Predicted note-event PNG", pred_events_png_path),
    ("Predicted Visual MIDI HTML", pred_html_path),
    ("Original input audio", audio_path),
]

if gt is not None:
    download_items.extend([
        ("Evaluation GT MIDI from cached rolls", gt_eval_midi_path),
        ("Evaluation GT MIDI audio WAV", gt_eval_wav_path),
        ("Prediction vs GT piano-roll PNG", gt_png_path),
        ("Frame diff PNG", diff_png_path),
        ("Predicted vs GT event overlay PNG", event_overlay_path),
    ])

if original_midi_path is not None and Path(original_midi_path).exists():
    download_items.extend([
        ("Original MAESTRO MIDI", original_midi_path),
        ("Original MAESTRO MIDI audio WAV", original_midi_wav_path),
        ("Original MIDI velocity+sustain PNG", original_sustain_png_path),
        ("Original Visual MIDI HTML", original_html_path),
    ])

buttons = []
for label, path in download_items:
    if path is None:
        continue
    path = Path(path)
    if not path.exists():
        continue
    btn = widgets.Button(description=label[:38], icon="download", layout=widgets.Layout(width="360px"))
    def _make_handler(p):
        def _handler(_):
            files.download(str(p))
        return _handler
    btn.on_click(_make_handler(path))
    buttons.append(btn)

if buttons:
    display(widgets.VBox(buttons))
else:
    print("No downloadable files found yet. Run the previous cells first.")


Download files


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>